In [1]:
!export HF_ENDPOINT=https://hf-mirror.com

In [2]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

In [3]:
import os
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ["HF_HOME"] = "/root/autodl-tmp/hf_cache"
os.environ["TRANSFORMERS_CACHE"] = "/root/autodl-tmp/hf_cache"

In [8]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

cache_dir = r"autodl-tmp/D:\models\hf_cache"

os.environ["HF_HOME"] = cache_dir
os.environ["HUGGINGFACE_HUB_CACHE"] = os.path.join(cache_dir, "hub")
os.environ["TRANSFORMERS_CACHE"] = os.path.join(cache_dir, "transformers")
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"   # 国内建议加这个

model_name = "Qwen/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    cache_dir=cache_dir,
    trust_remote_code=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    cache_dir=cache_dir,
    trust_remote_code=True,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

print("Qwen2.5-7B-Instruct 加载成功")

Loading weights: 100%|██████████| 339/339 [00:07<00:00, 44.78it/s]


Qwen2.5-7B-Instruct 加载成功


In [5]:
import os
import pandas as pd

data_dir = "/root/autodl-tmp/FakeNewsNet/fakenewsnet"

gossipcop_fake = pd.read_csv(os.path.join(data_dir, "gossipcop_fake.csv"))
gossipcop_real = pd.read_csv(os.path.join(data_dir, "gossipcop_real.csv"))
politifact_fake = pd.read_csv(os.path.join(data_dir, "politifact_fake.csv"))
politifact_real = pd.read_csv(os.path.join(data_dir, "politifact_real.csv"))

print("gossipcop_fake:", gossipcop_fake.shape)
print("gossipcop_real:", gossipcop_real.shape)
print("politifact_fake:", politifact_fake.shape)
print("politifact_real:", politifact_real.shape)
gossipcop_fake["label"] = 0
gossipcop_real["label"] = 1
politifact_fake["label"] = 0
politifact_real["label"] = 1

gossipcop_fake["source"] = "gossipcop"
gossipcop_real["source"] = "gossipcop"
politifact_fake["source"] = "politifact"
politifact_real["source"] = "politifact"

df = pd.concat(
    [gossipcop_fake, gossipcop_real, politifact_fake, politifact_real],
    ignore_index=True
)

print("总数据量:", len(df))
print("列名:", df.columns.tolist())
print(df["label"].value_counts())
candidate_cols = ["text", "title", "content", "headline"]
text_col = None

for col in candidate_cols:
    if col in df.columns:
        text_col = col
        break

if text_col is None:
    raise ValueError(f"没有找到可用文本列，当前列名为: {df.columns.tolist()}")

print("使用文本列:", text_col)

df = df[[text_col, "label", "source"]].dropna()
df[text_col] = df[text_col].astype(str).str.strip()
df = df[df[text_col] != ""]
df = df.drop_duplicates(subset=[text_col]).reset_index(drop=True)

print("清洗后数据量:", len(df))
print(df["label"].value_counts())
df.head()
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.1,
    random_state=42,
    stratify=df["label"]
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("训练/知识库集:", train_df.shape)
print("测试集:", test_df.shape)
save_dir = "/root/autodl-tmp/FakeNewsNet/splits"
os.makedirs(save_dir, exist_ok=True)

train_df.to_csv(os.path.join(save_dir, "fakenewsnet_train_90.csv"), index=False)
test_df.to_csv(os.path.join(save_dir, "fakenewsnet_test_10.csv"), index=False)

print("已保存到:", save_dir)

gossipcop_fake: (5323, 4)
gossipcop_real: (16817, 4)
politifact_fake: (432, 4)
politifact_real: (624, 4)
总数据量: 23196
列名: ['id', 'news_url', 'title', 'tweet_ids', 'label', 'source']
label
1    17441
0     5755
Name: count, dtype: int64
使用文本列: title
清洗后数据量: 21724
label
1    16402
0     5322
Name: count, dtype: int64
训练/知识库集: (19551, 3)
测试集: (2173, 3)
已保存到: /root/autodl-tmp/FakeNewsNet/splits


In [6]:
import os
import pandas as pd

train_path = "/root/autodl-tmp/FakeNewsNet/splits/fakenewsnet_train_90.csv"
test_path = "/root/autodl-tmp/FakeNewsNet/splits/fakenewsnet_test_10.csv"

print("train exists:", os.path.exists(train_path))
print("test exists:", os.path.exists(test_path))

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print("train shape:", train_df.shape)
print("test shape:", test_df.shape)

display(train_df.head())
display(test_df.head())

train exists: True
test exists: True
train shape: (19551, 3)
test shape: (2173, 3)


,title,label,source
0,Andrew Garfield's First Kiss Was with '30 Girl...,1,gossipcop
1,Harry Styles Covers Kanye West During First So...,1,gossipcop
2,Is Lady Gaga Engaged? She & Christian Carino A...,0,gossipcop
3,Meghan Markle Spotted Her Old Drama Teacher Du...,1,gossipcop
4,Selena Gomez Had The Perfect Response To News ...,0,gossipcop


,title,label,source
0,Kendra wilkinson-baskett confronts her mom's s...,1,gossipcop
1,Naturi Naughton,1,gossipcop
2,Shawn Mendes Looks Back on “Unreal” Performanc...,1,gossipcop
3,New Parents Kylie Jenner and Travis Scott Step...,1,gossipcop
4,"Nicole Williams: her new entrepreneur role, he...",1,gossipcop


In [7]:
import os
import uuid
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
import chromadb

db_dir = "/root/autodl-tmp/FakeNewsNet/chroma_db_bge"
collection_name = "fakenewsnet_90_bge"

os.makedirs(db_dir, exist_ok=True)

# 只保留需要的列
train_df = train_df[[text_col, "label", "source"]].dropna().copy()
train_df[text_col] = train_df[text_col].astype(str).str.strip()
train_df = train_df[train_df[text_col] != ""].drop_duplicates(subset=[text_col]).reset_index(drop=True)

print("用于建库的训练样本数:", len(train_df))

# embedding 模型
embed_model = SentenceTransformer("BAAI/bge-base-en-v1.5")

# 初始化 Chroma
client = chromadb.PersistentClient(path=db_dir)

existing = [c.name for c in client.list_collections()]
if collection_name in existing:
    client.delete_collection(collection_name)
    print("已删除旧 collection:", collection_name)

collection = client.create_collection(
    name=collection_name,
    metadata={"description": "FakeNewsNet 90 percent train set with BGE embeddings"}
)

print("已创建新 collection:", collection_name)

batch_size = 128

documents = []
documents_for_embedding = []
metadatas = []
ids = []

for _, row in train_df.iterrows():
    text = row[text_col]
    label = int(row["label"])
    source = row["source"]

    documents.append(text)
    documents_for_embedding.append(f"News document: {text}")
    metadatas.append({
        "label": label,
        "label_name": "real" if label == 1 else "fake",
        "source": source
    })
    ids.append(str(uuid.uuid4()))

for start in tqdm(range(0, len(documents), batch_size)):
    end = start + batch_size

    batch_docs = documents[start:end]
    batch_docs_for_embedding = documents_for_embedding[start:end]
    batch_meta = metadatas[start:end]
    batch_ids = ids[start:end]

    batch_embeds = embed_model.encode(
        batch_docs_for_embedding,
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=True
    ).tolist()

    collection.add(
        ids=batch_ids,
        documents=batch_docs,
        metadatas=batch_meta,
        embeddings=batch_embeds
    )

print("建库完成")
print("collection count =", collection.count())
print("db_dir =", db_dir)

用于建库的训练样本数: 19551


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4283.59it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


已删除旧 collection: fakenewsnet_90_bge
已创建新 collection: fakenewsnet_90_bge


100%|██████████| 153/153 [00:26<00:00,  5.75it/s]

建库完成
collection count = 19551
db_dir = /root/autodl-tmp/FakeNewsNet/chroma_db_bge


In [9]:
import os
import re
import torch
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import chromadb

# =========================================================
# 1. 路径配置
# =========================================================
data_dir = "/root/autodl-tmp/FakeNewsNet"
test_file = os.path.join(data_dir, "splits", "fakenewsnet_test_10.csv")

db_dir = os.path.join(data_dir, "chroma_db_bge")
collection_name = "fakenewsnet_90_bge"



sample_size = 2173
top_k = 5
save_path = os.path.join(data_dir, "qwen_7b_rag_bge_test10_results.csv")

# =========================================================
# 2. 基本检查
# =========================================================
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("CUDA 不可用，当前代码要求 embedding 走 GPU。")
print("gpu:", torch.cuda.get_device_name(0))

# =========================================================
# 3. 读取测试集
# =========================================================
test_df = pd.read_csv(test_file)
print("测试集大小:", len(test_df))
print("测试集列名:", test_df.columns.tolist())

candidate_cols = ["text", "title", "content", "headline"]
text_col = None
for col in candidate_cols:
    if col in test_df.columns:
        text_col = col
        break

if text_col is None:
    raise ValueError(f"没找到文本列，当前列名: {test_df.columns.tolist()}")

print("使用文本列:", text_col)

test_df = test_df[[text_col, "label", "source"]].dropna().reset_index(drop=True)
test_df[text_col] = test_df[text_col].astype(str).str.strip()
test_df = test_df[test_df[text_col] != ""].reset_index(drop=True)

print("清洗后测试集大小:", len(test_df))

if len(test_df) > sample_size:
    eval_df = test_df.sample(n=sample_size, random_state=42).reset_index(drop=True)
else:
    eval_df = test_df.copy()

print("实际评估样本数:", len(eval_df))
print(eval_df["label"].value_counts())
display(eval_df.head())

# =========================================================
# 4. 加载 ChromaDB 向量库
# =========================================================
client = chromadb.PersistentClient(path=db_dir)
collection = client.get_collection(collection_name)

print("向量库已连接")
print("collection count =", collection.count())

# =========================================================
# 5. 加载 embedding 模型（GPU）
# =========================================================
embed_model = SentenceTransformer("BAAI/bge-base-en-v1.5", device="cuda")
print("embedding model device: cuda")

# 可选：单条测试，确认 embedding GPU 正常
_test_emb = embed_model.encode(
    ["Represent this news query for retrieving relevant similar news: test sentence"],
    normalize_embeddings=True,
    convert_to_numpy=True
)
print("embedding smoke test ok, shape:", _test_emb.shape)

# =========================================================
# 6. 加载 Qwen2.5-7B-Instruct（4bit）
# =========================================================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_name)


# =========================================================
# 7. 检索函数
# =========================================================
def retrieve_docs(query_text, top_k=5):
    query_for_embedding = f"Represent this news query for retrieving relevant similar news: {query_text}"

    query_embedding = embed_model.encode(
        [query_for_embedding],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k
    )

    docs = results["documents"][0]
    metas = results["metadatas"][0]
    dists = results["distances"][0] if "distances" in results else [None] * len(docs)

    retrieved = []
    for doc, meta, dist in zip(docs, metas, dists):
        retrieved.append({
            "text": doc,
            "label": meta.get("label_name", "unknown"),
            "source": meta.get("source", "unknown"),
            "distance": dist
        })

    return retrieved

# =========================================================
# 8. 检索投票兜底
# =========================================================
def fallback_by_retrieval(retrieved_docs):
    if len(retrieved_docs) == 0:
        return 0

    fake_score = 0.0
    real_score = 0.0

    for d in retrieved_docs:
        dist = d["distance"] if d["distance"] is not None else 1.0
        weight = 1.0 / (dist + 1e-6)

        if d["label"] == "fake":
            fake_score += weight
        elif d["label"] == "real":
            real_score += weight

    return 0 if fake_score >= real_score else 1

# =========================================================
# 9. 构造 RAG Prompt
# =========================================================
def build_rag_prompt(query_text, retrieved_docs):
    context_parts = []

    for i, item in enumerate(retrieved_docs):
        distance_str = "None" if item["distance"] is None else f"{item['distance']:.6f}"
        context_parts.append(
            f"[Reference {i+1}]\n"
            f"Source: {item['source']}\n"
            f"Verified label: {item['label']}\n"
            f"Similarity distance: {distance_str}\n"
            f"Text: {item['text']}\n"
        )

    context = "\n".join(context_parts)

    prompt = (
        "You are an expert fake news judge.\n"
        "You are given one target news item and several retrieved news references from a database.\n"
        "Each reference has a verified ground-truth label: fake or real.\n\n"
        "Your job:\n"
        "1. For each retrieved reference, decide whether it is truly relevant to the target news.\n"
        "2. If relevant, judge whether it provides positive evidence or negative evidence for the target label.\n"
        "   - Positive evidence means the reference supports a similar truth pattern for the target news.\n"
        "   - Negative evidence means the reference is related on the surface but should not be used as supporting evidence.\n"
        "3. Ignore irrelevant or misleading references.\n"
        "4. Then make a final decision on whether the target news is fake or real.\n\n"
        "Important rules:\n"
        "- Do not trust every retrieved reference blindly.\n"
        "- First judge relevance.\n"
        "- Then judge whether the evidence is positive or negative.\n"
        "- Use only the truly relevant positive evidence strongly.\n"
        "- If evidence conflicts, weigh the more relevant references more heavily.\n\n"
        "Output format must be exactly:\n"
        "analysis: <brief reasoning summarizing which references are relevant, whether they are positive or negative evidence, and why>\n"
        "answer: fake\n"
        "or\n"
        "analysis: <brief reasoning summarizing which references are relevant, whether they are positive or negative evidence, and why>\n"
        "answer: real\n\n"
        "Do not output anything after the answer line.\n\n"
        f"Retrieved references:\n{context}\n\n"
        f"Target news:\n{query_text}\n"
    )

    return prompt

# =========================================================
# 10. 解析模型输出
# =========================================================
def parse_prediction(output_text, retrieved_docs=None):
    text = output_text.strip().lower()

    match = re.search(r"answer\s*:\s*(fake|real)", text)
    if match:
        return 0 if match.group(1) == "fake" else 1

    if "fake" in text and "real" not in text:
        return 0
    if "real" in text and "fake" not in text:
        return 1

    fake_pos = text.find("fake") if "fake" in text else 10**9
    real_pos = text.find("real") if "real" in text else 10**9

    if fake_pos < real_pos:
        return 0
    if real_pos < fake_pos:
        return 1

    if retrieved_docs is not None:
        return fallback_by_retrieval(retrieved_docs)

    return 0

# =========================================================
# 11. 单条预测
# =========================================================
def predict_one_with_rag(news_text, top_k=5):
    retrieved_docs = retrieve_docs(news_text, top_k=top_k)
    prompt = build_rag_prompt(news_text[:2000], retrieved_docs)

    messages = [
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=160,
            do_sample=False
        )

    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    pred = parse_prediction(response, retrieved_docs)

    return response, pred, retrieved_docs

# =========================================================
# 12. 批量评估
# =========================================================
y_true = []
y_pred = []
raw_outputs = []
retrieved_texts = []

for i, row in tqdm(eval_df.iterrows(), total=len(eval_df)):
    news_text = row[text_col]
    true_label = int(row["label"])

    try:
        raw_output, pred_label, retrieved_docs = predict_one_with_rag(news_text, top_k=top_k)

        retrieved_joined = "\n\n".join(
            [
                f"[{j+1}] label={d['label']}, source={d['source']}, distance={d['distance']}, text={d['text'][:300]}"
                for j, d in enumerate(retrieved_docs)
            ]
        )
    except Exception as e:
        raw_output = f"ERROR: {e}"
        retrieved_docs = []
        pred_label = fallback_by_retrieval(retrieved_docs)
        retrieved_joined = ""

    y_true.append(true_label)
    y_pred.append(pred_label)
    raw_outputs.append(raw_output)
    retrieved_texts.append(retrieved_joined)

    print(f"\n[{i+1}/{len(eval_df)}]")
    print("true =", true_label, "pred =", pred_label)
    print("raw_output =", raw_output)

# =========================================================
# 13. 结果统计
# =========================================================
result_df = eval_df.copy()
result_df["pred"] = y_pred
result_df["raw_output"] = raw_outputs
result_df["retrieved_docs"] = retrieved_texts

result_df["pred"] = result_df["pred"].astype(int)

acc = accuracy_score(result_df["label"], result_df["pred"])

print("\n总样本数:", len(result_df))
print("Accuracy:", acc)
print("\nClassification Report:")
print(classification_report(result_df["label"], result_df["pred"], digits=4))

# =========================================================
# 14. 保存结果
# =========================================================
result_df.to_csv(save_path, index=False)

print("\n结果已保存到:", save_path)
display(result_df.head())

torch: 2.11.0+cu130
cuda available: True
gpu: NVIDIA GeForce RTX 5090
测试集大小: 2173
测试集列名: ['title', 'label', 'source']
使用文本列: title
清洗后测试集大小: 2173
实际评估样本数: 1000
label
1    756
0    244
Name: count, dtype: int64


,title,label,source
0,Jersey Shore Star Ronnie Ortiz-Magro's Ex Jen ...,1,gossipcop
1,Ben Affleck looking for family friendly role,0,gossipcop
2,Jennifer Aniston and Courteney Cox: Best Frien...,1,gossipcop
3,Nicole Kidman Reveals the Secret to Her 12-Yea...,1,gossipcop
4,Trump’s Top Scientist Pick: “Scientists Are Ju...,0,politifact


向量库已连接
collection count = 19551


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4056.92it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


embedding model device: cuda
embedding smoke test ok, shape: (1, 768)


...

100%|█████████▉| 999/1000 [23:30<00:01,  1.39s/it]


[999/1000]
true = 1 pred = 1
raw_output = analysis: None of the retrieved references are directly relevant to the target news item about Jamie Scott from "One Tree Hill." The references are mostly about Kylie Jenner and Travis Scott's relationship and pregnancy rumors. Since there is no relevant information provided that could support or refute the claim about Jamie Scott, we cannot make a determination based on these references alone.

answer: real


100%|██████████| 1000/1000 [23:31<00:00,  1.41s/it]


[1000/1000]
true = 1 pred = 1
raw_output = analysis: Reference 1 and Reference 3 are relevant as they discuss government spending and budget issues, aligning with the target news. Both provide positive evidence that the target news is likely real, as they highlight government borrowing and spending. Reference 4 and Reference 5 are not relevant as they discuss different topics and are labeled as fake, which does not affect our judgment. Given the strong alignment of References 1 and 3 with the target news and their real labels, the target news is supported by positive evidence.

answer: real

总样本数: 1000
Accuracy: 0.702

Classification Report:
              precision    recall  f1-score   support

           0     0.3738    0.3279    0.3493       244
           1     0.7913    0.8228    0.8067       756

    accuracy                         0.7020      1000
   macro avg     0.5826    0.5753    0.5780      1000
weighted avg     0.6895    0.7020    0.6951      1000


结果已保存到: /root/autodl-

,title,label,source,pred,raw_output,retrieved_docs
0,Jersey Shore Star Ronnie Ortiz-Magro's Ex Jen ...,1,gossipcop,1,analysis: Reference 1 is relevant and provides...,"[1] label=real, source=gossipcop, distance=0.1..."
1,Ben Affleck looking for family friendly role,0,gossipcop,1,analysis: Reference 2 and Reference 4 are rele...,"[1] label=real, source=gossipcop, distance=0.6..."
2,Jennifer Aniston and Courteney Cox: Best Frien...,1,gossipcop,1,analysis: Reference 2 is relevant and provides...,"[1] label=fake, source=gossipcop, distance=0.4..."
3,Nicole Kidman Reveals the Secret to Her 12-Yea...,1,gossipcop,1,analysis: Reference 4 is the only relevant and...,"[1] label=fake, source=gossipcop, distance=0.3..."
4,Trump’s Top Scientist Pick: “Scientists Are Ju...,0,politifact,0,analysis: Reference 4 is relevant and provides...,"[1] label=fake, source=politifact, distance=0...."


In [11]:
import os
import re
import shutil
import torch
import pandas as pd
import chromadb

from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# =========================================================
# 1. 路径配置
# =========================================================
ROOT_DIR = "/root/autodl-tmp"
DATASET_NAME = "CT22_en_1C_harmful"

TRAIN_PATH = "autodl-tmp/CT22_en_1C_harmful_train.tsv"
TEST_PATH = "autodl-tmp/CT22_en_1C_harmful_test_gold.tsv"

OUTPUT_DIR = os.path.join(ROOT_DIR, f"{DATASET_NAME}_rag_output")
DB_DIR = os.path.join(OUTPUT_DIR, "chroma_db")
COLLECTION_NAME = f"{DATASET_NAME}_bge"
SAVE_PATH = os.path.join(OUTPUT_DIR, f"{DATASET_NAME}_rag_results.csv")

EMBED_MODEL_NAME = "BAAI/bge-base-en-v1.5"

TOP_K = 5
MAX_NEW_TOKENS = 600
BATCH_SIZE = 256

os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================================================
# 2. 基本检查
# =========================================================
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("CUDA 不可用")
print("gpu:", torch.cuda.get_device_name(0))

# =========================================================
# 3. 读取 train / test
# =========================================================
train_df = pd.read_csv(TRAIN_PATH, sep="\t")
test_df = pd.read_csv(TEST_PATH, sep="\t")

print("train columns:", train_df.columns.tolist())
print("test columns:", test_df.columns.tolist())

# 统一字段
required_cols = ["tweet_text", "class_label"]
for col in required_cols:
    if col not in train_df.columns:
        raise ValueError(f"训练集缺少列: {col}")
    if col not in test_df.columns:
        raise ValueError(f"测试集缺少列: {col}")

train_df = train_df[["tweet_text", "class_label", "topic"]].copy()
test_df = test_df[["tweet_text", "class_label", "topic"]].copy()

train_df = train_df.rename(columns={"tweet_text": "text", "class_label": "label", "topic": "source"})
test_df = test_df.rename(columns={"tweet_text": "text", "class_label": "label", "topic": "source"})

train_df = train_df.dropna(subset=["text", "label"]).reset_index(drop=True)
test_df = test_df.dropna(subset=["text", "label"]).reset_index(drop=True)

train_df["text"] = train_df["text"].astype(str).str.strip()
test_df["text"] = test_df["text"].astype(str).str.strip()

train_df = train_df[train_df["text"] != ""].reset_index(drop=True)
test_df = test_df[test_df["text"] != ""].reset_index(drop=True)

train_df["label"] = train_df["label"].astype(int)
test_df["label"] = test_df["label"].astype(int)

print("train size:", len(train_df))
print("test size:", len(test_df))
print("train label dist:")
print(train_df["label"].value_counts())
print("test label dist:")
print(test_df["label"].value_counts())

# =========================================================
# 4. 加载 embedding 模型
# =========================================================
embed_model = SentenceTransformer(EMBED_MODEL_NAME, device="cuda")
print("embedding model device: cuda")

# smoke test
_test_emb = embed_model.encode(
    ["Represent this tweet for retrieving relevant harmfulness examples: test sentence"],
    normalize_embeddings=True,
    convert_to_numpy=True
)
print("embedding smoke test ok, shape:", _test_emb.shape)

# =========================================================
# 5. 加载 Qwen 4bit
# =========================================================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)



print("Qwen loaded")

# =========================================================
# 6. 建 Chroma 向量库（只用训练集）
# 标签约定：
# 0 = non-harmful
# 1 = harmful
# # =========================================================
# if os.path.exists(DB_DIR):
#     shutil.rmtree(DB_DIR)

# client = chromadb.PersistentClient(path=DB_DIR)
# collection = client.create_collection(
#     name=COLLECTION_NAME,
#     metadata={"hnsw:space": "cosine"}
# )

# texts = train_df["text"].tolist()
# labels = train_df["label"].tolist()
# sources = train_df["source"].tolist()
# ids = [f"doc_{i}" for i in range(len(texts))]

# for start in tqdm(range(0, len(texts), BATCH_SIZE), desc="building chroma"):
#     end = min(start + BATCH_SIZE, len(texts))
#     batch_texts = texts[start:end]

#     batch_embeddings = embed_model.encode(
#         [f"Represent this tweet for retrieving relevant harmfulness examples: {x}" for x in batch_texts],
#         normalize_embeddings=True,
#         convert_to_numpy=True
#     ).tolist()

#     batch_metas = []
#     for lab, src in zip(labels[start:end], sources[start:end]):
#         batch_metas.append({
#             "label": int(lab),
#             "label_name": "harmful" if int(lab) == 1 else "non-harmful",
#             "source": str(src)
#         })

#     collection.add(
#         ids=ids[start:end],
#         documents=batch_texts,
#         embeddings=batch_embeddings,
#         metadatas=batch_metas
#     )

# print("collection count =", collection.count())

# =========================================================
# 7. 检索函数
# =========================================================
def retrieve_docs(query_text, top_k=5):
    query_embedding = embed_model.encode(
        [f"Represent this tweet for retrieving relevant harmfulness examples: {query_text}"],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k
    )

    docs = results["documents"][0]
    metas = results["metadatas"][0]
    dists = results["distances"][0] if "distances" in results else [None] * len(docs)

    retrieved = []
    for doc, meta, dist in zip(docs, metas, dists):
        retrieved.append({
            "text": doc,
            "label": meta.get("label_name", "unknown"),
            "source": meta.get("source", "unknown"),
            "distance": dist
        })

    return retrieved

# =========================================================
# 8. 检索兜底
# =========================================================
def fallback_by_retrieval(retrieved_docs):
    if len(retrieved_docs) == 0:
        return 0

    harmful_score = 0.0
    non_harmful_score = 0.0

    for d in retrieved_docs:
        dist = d["distance"] if d["distance"] is not None else 1.0
        weight = 1.0 / (dist + 1e-6)

        if d["label"] == "harmful":
            harmful_score += weight
        elif d["label"] == "non-harmful":
            non_harmful_score += weight

    return 1 if harmful_score > non_harmful_score else 0

# =========================================================
# 9. Prompt
# =========================================================
def build_rag_prompt(query_text, retrieved_docs):
    context_parts = []

    for i, item in enumerate(retrieved_docs):
        dist_str = "None" if item["distance"] is None else f"{item['distance']:.6f}"
        context_parts.append(
            f"[Reference {i+1}]\n"
            f"Source: {item['source']}\n"
            f"Verified label: {item['label']}\n"
            f"Similarity distance: {dist_str}\n"
            f"Text: {item['text']}\n"
        )

    context = "\n".join(context_parts)

    return f"""
You are a careful harmful content detection assistant.

Your task is to classify the TARGET TWEET as either:
- harmful
- non-harmful

You are given retrieved references with verified labels, but these references may be noisy or only superficially similar.
Do not trust them automatically.

Instructions:
1. For each reference, decide whether it is:
   - relevant_positive
   - relevant_negative
   - irrelevant
2. Only relevant_positive references can be used as strong evidence.
3. Ignore references that only overlap in topic, wording, or style.
4. If evidence is weak or conflicting, make a cautious judgment based on the target tweet itself.
5. Prefer the strongest matching evidence, not the largest number of references.

Guidance:
- harmful: promotes or spreads harmful misinformation, dangerous claims, or harmful misleading content
- non-harmful: neutral, factual, conversational, or harmless discussion

Output exactly in this format:

analysis:
Reference 1: <relevant_positive / relevant_negative / irrelevant> - <short reason>
Reference 2: <relevant_positive / relevant_negative / irrelevant> - <short reason>
Reference 3: <relevant_positive / relevant_negative / irrelevant> - <short reason>
Reference 4: <relevant_positive / relevant_negative / irrelevant> - <short reason>
Reference 5: <relevant_positive / relevant_negative / irrelevant> - <short reason>
final reasoning: <brief paragraph>

answer: <harmful or non-harmful>

Retrieved references:
{context}

Target tweet:
{query_text}
"""

# =========================================================
# 10. 输出解析
# =========================================================
def parse_prediction(output_text, retrieved_docs=None):
    text = output_text.strip().lower()

    match = re.search(r"answer\s*:\s*(harmful|non-harmful)", text)
    if match:
        return 1 if match.group(1) == "harmful" else 0

    if "non-harmful" in text:
        return 0
    if "harmful" in text:
        return 1

    if retrieved_docs is not None:
        return fallback_by_retrieval(retrieved_docs)

    return 0

# =========================================================
# 11. 单条预测
# =========================================================
def predict_one_with_rag(tweet_text, top_k=5):
    retrieved_docs = retrieve_docs(tweet_text, top_k=top_k)
    prompt = build_rag_prompt(tweet_text[:2000], retrieved_docs)

    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=None,
            top_p=None
        )

    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    pred = parse_prediction(response, retrieved_docs)

    return response, pred, retrieved_docs

# =========================================================
# 12. 测试集评估
# =========================================================
y_true = []
y_pred = []
raw_outputs = []
retrieved_texts = []

for i, row in tqdm(test_df.iterrows(), total=len(test_df), desc="evaluating"):
    tweet_text = str(row["text"])
    true_label = int(row["label"])

    try:
        raw_output, pred_label, retrieved_docs = predict_one_with_rag(tweet_text, top_k=TOP_K)

        retrieved_joined = "\n\n".join([
            f"[{j+1}] label={d['label']}, source={d['source']}, distance={d['distance']}, text={d['text'][:300]}"
            for j, d in enumerate(retrieved_docs)
        ])
    except Exception as e:
        raw_output = f"ERROR: {e}"
        pred_label = 0
        retrieved_joined = ""

    y_true.append(true_label)
    y_pred.append(pred_label)
    raw_outputs.append(raw_output)
    retrieved_texts.append(retrieved_joined)

    print(f"\n[{i+1}/{len(test_df)}]")
    print("true =", true_label, "pred =", pred_label)
    print("raw_output =", raw_output[:500])

# =========================================================
# 13. 结果统计
# =========================================================
result_df = test_df.copy()
result_df["pred"] = y_pred
result_df["raw_output"] = raw_outputs
result_df["retrieved_docs"] = retrieved_texts

acc = accuracy_score(result_df["label"], result_df["pred"])

print("\n总样本数:", len(result_df))
print("Accuracy:", acc)
print("\nClassification Report:")
print(classification_report(result_df["label"], result_df["pred"], digits=4))

# =========================================================
# 14. 保存结果
# =========================================================
result_df.to_csv(SAVE_PATH, index=False)
print("\n结果已保存到:", SAVE_PATH)

torch: 2.11.0+cu130
cuda available: True
gpu: NVIDIA GeForce RTX 5090
train columns: ['topic', 'tweet_id', 'tweet_url', 'tweet_text', 'class_label']
test columns: ['topic', 'tweet_id', 'tweet_url', 'tweet_text', 'class_label']
train size: 3323
test size: 251
train label dist:
label
0    3031
1     292
Name: count, dtype: int64
test label dist:
label
0    211
1     40
Name: count, dtype: int64


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4286.96it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


embedding model device: cuda
embedding smoke test ok, shape: (1, 768)
Qwen loaded


evaluating:   0%|          | 1/251 [00:02<08:55,  2.14s/it]


[1/251]
true = 0 pred = 0
raw_output = analysis:
Reference 1: irrelevant - No direct relation to the target tweet.
Reference 2: irrelevant - No direct relation to the target tweet.
Reference 3: irrelevant - No direct relation to the target tweet.
Reference 4: irrelevant - No direct relation to the target tweet.
Reference 5: irrelevant - No direct relation to the target tweet.
final reasoning: The target tweet makes a humorous statement suggesting that characters from Phineas and Ferb would have developed a vaccine much faster than th


...


[249/251]
true = 0 pred = 0
raw_output = analysis:
Reference 1: irrelevant - The reference discusses vaccination as a safe and effective way to protect against frailty and provides a link to a related article.
Reference 2: irrelevant - This reference emphasizes the importance of vaccination in protecting against serious illness and being there for friends and family, also providing a link to an article.
Reference 3: relevant_negative - The reference contains a mistrustful and potentially harmful claim about government denial of vaccine


evaluating: 100%|█████████▉| 250/251 [11:35<00:03,  3.09s/it]


[250/251]
true = 0 pred = 0
raw_output = analysis:
Reference 1: irrelevant - The reference discusses patent issues related to COVID-19 vaccines and does not directly relate to the OSHA vaccine mandate.
Reference 2: irrelevant - This reference talks about the GOP blocking a coronavirus bill, which is not directly related to the OSHA vaccine mandate.
Reference 3: irrelevant - This reference expresses concerns about the premature approval of COVID-19 vaccines, which is unrelated to the OSHA vaccine mandate.
Reference 4: irrelevant - This 


evaluating: 100%|██████████| 251/251 [11:37<00:00,  2.78s/it]


[251/251]
true = 0 pred = 0
raw_output = analysis:
Reference 1: irrelevant - Overlaps in topic but does not provide direct evidence.
Reference 2: irrelevant - Overlaps in topic but does not provide direct evidence.
Reference 3: irrelevant - Overlaps in topic but does not provide direct evidence.
Reference 4: irrelevant - Overlaps in topic but does not provide direct evidence.
Reference 5: irrelevant - Overlaps in topic but does not provide direct evidence.
final reasoning: The target tweet discusses the global approach to vaccination a

总样本数: 251
Accuracy: 0.7290836653386454

Classification Report:
              precision    recall  f1-score   support

           0     0.8950    0.7678    0.8265       211
           1     0.3000    0.5250    0.3818        40

    accuracy                         0.7291       251
   macro avg     0.5975    0.6464    0.6042       251
weighted avg     0.8002    0.7291    0.7557       251


结果已保存到: /root/autodl-tmp/CT22_en_1C_harmful_rag_output/CT22_en_1C

In [14]:
import os
import re
import shutil
import torch
import pandas as pd
import chromadb

from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# =========================================================
# 1. 路径配置
# =========================================================
ROOT_DIR = "/root/autodl-tmp"
DATASET_NAME = "train_test_tsv_rag"

TRAIN_PATH = "autodl-tmp/train.tsv"
TEST_PATH = "autodl-tmp/test.tsv"

OUTPUT_DIR = os.path.join(ROOT_DIR, f"{DATASET_NAME}_output")
DB_DIR = os.path.join(OUTPUT_DIR, "chroma_db")
COLLECTION_NAME = f"{DATASET_NAME}_bge"
SAVE_PATH = os.path.join(OUTPUT_DIR, "rag_results.csv")

EMBED_MODEL_NAME = "BAAI/bge-base-en-v1.5"

TOP_K = 5
MAX_NEW_TOKENS = 220
BATCH_SIZE = 128

os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================================================
# 2. 基本检查
# =========================================================
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("CUDA 不可用")
print("gpu:", torch.cuda.get_device_name(0))

# =========================================================
# 3. 读取 train / test
# =========================================================
train_df = pd.read_csv(TRAIN_PATH, sep="\t")
test_df = pd.read_csv(TEST_PATH, sep="\t")

print("train columns:", train_df.columns.tolist())
print("test columns:", test_df.columns.tolist())

required_cols = ["sentence", "label"]
for col in required_cols:
    if col not in train_df.columns:
        raise ValueError(f"训练集缺少列: {col}")
    if col not in test_df.columns:
        raise ValueError(f"测试集缺少列: {col}")

train_df = train_df[["sentence", "label"]].copy()
test_df = test_df[["sentence", "label"]].copy()

train_df = train_df.rename(columns={"sentence": "text", "label": "raw_label"})
test_df = test_df.rename(columns={"sentence": "text", "label": "raw_label"})

train_df = train_df.dropna(subset=["text", "raw_label"]).reset_index(drop=True)
test_df = test_df.dropna(subset=["text", "raw_label"]).reset_index(drop=True)

train_df["text"] = train_df["text"].astype(str).str.strip()
test_df["text"] = test_df["text"].astype(str).str.strip()

train_df = train_df[train_df["text"] != ""].reset_index(drop=True)
test_df = test_df[test_df["text"] != ""].reset_index(drop=True)

train_df["raw_label"] = train_df["raw_label"].astype(int)
test_df["raw_label"] = test_df["raw_label"].astype(int)

# =========================================================
# 4. 标签映射
# 内部统一：
#   0 = fake
#   1 = real
#
# 当前先假设原始:
#   1 -> fake
#   0 -> real
#
# 如果你确认相反，把下面两行改成：
#   return 0 if x == 0 else 1
# =========================================================
def map_label(x: int) -> int:
    return 0 if x == 1 else 1

train_df["label"] = train_df["raw_label"].apply(map_label).astype(int)
test_df["label"] = test_df["raw_label"].apply(map_label).astype(int)

train_df["source"] = "train.tsv"
test_df["source"] = "test.tsv"

print("train size:", len(train_df))
print("test size:", len(test_df))
print("train raw label dist:")
print(train_df["raw_label"].value_counts())
print("test raw label dist:")
print(test_df["raw_label"].value_counts())
print("train mapped label dist (0=fake,1=real):")
print(train_df["label"].value_counts())
print("test mapped label dist (0=fake,1=real):")
print(test_df["label"].value_counts())

# =========================================================
# 5. 加载 embedding 模型
# =========================================================
embed_model = SentenceTransformer(EMBED_MODEL_NAME, device="cuda")
print("embedding model device: cuda")

_test_emb = embed_model.encode(
    ["Represent this news query for retrieving relevant similar news: test sentence"],
    normalize_embeddings=True,
    convert_to_numpy=True
)
print("embedding smoke test ok, shape:", _test_emb.shape)

# =========================================================
# 6. 加载 Qwen 4bit
# =========================================================
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_use_double_quant=True,
#     bnb_4bit_compute_dtype=torch.float16
# )

tokenizer = AutoTokenizer.from_pretrained(QWEN_PATH, trust_remote_code=True)



print("Qwen loaded")

# =========================================================
# 7. 建库（只用 train）
# =========================================================
if os.path.exists(DB_DIR):
    shutil.rmtree(DB_DIR)

client = chromadb.PersistentClient(path=DB_DIR)
collection = client.create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"}
)

texts = train_df["text"].tolist()
labels = train_df["label"].tolist()
sources = train_df["source"].tolist()
ids = [f"doc_{i}" for i in range(len(texts))]

for start in tqdm(range(0, len(texts), BATCH_SIZE), desc="building chroma"):
    end = min(start + BATCH_SIZE, len(texts))
    batch_texts = texts[start:end]

    batch_embeddings = embed_model.encode(
        [f"Represent this news query for retrieving relevant similar news: {x}" for x in batch_texts],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).tolist()

    batch_metas = []
    for lab, src in zip(labels[start:end], sources[start:end]):
        batch_metas.append({
            "label": int(lab),
            "label_name": "fake" if int(lab) == 0 else "real",
            "source": str(src)
        })

    collection.add(
        ids=ids[start:end],
        documents=batch_texts,
        embeddings=batch_embeddings,
        metadatas=batch_metas
    )

print("collection count =", collection.count())

# =========================================================
# 8. 检索
# =========================================================
def retrieve_docs(query_text, top_k=5):
    query_embedding = embed_model.encode(
        [f"Represent this news query for retrieving relevant similar news: {query_text}"],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k
    )

    docs = results["documents"][0]
    metas = results["metadatas"][0]
    dists = results["distances"][0] if "distances" in results else [None] * len(docs)

    retrieved = []
    for doc, meta, dist in zip(docs, metas, dists):
        retrieved.append({
            "text": doc,
            "label": meta.get("label_name", "unknown"),
            "source": meta.get("source", "unknown"),
            "distance": dist
        })

    return retrieved

# =========================================================
# 9. 检索兜底
# =========================================================
def fallback_by_retrieval(retrieved_docs):
    if len(retrieved_docs) == 0:
        return 0

    fake_score = 0.0
    real_score = 0.0

    for d in retrieved_docs:
        dist = d["distance"] if d["distance"] is not None else 1.0
        weight = 1.0 / (dist + 1e-6)

        if d["label"] == "fake":
            fake_score += weight
        elif d["label"] == "real":
            real_score += weight

    return 0 if fake_score >= real_score else 1

# =========================================================
# 10. Prompt
# =========================================================
def build_rag_prompt(query_text, retrieved_docs):
    context_parts = []

    for i, item in enumerate(retrieved_docs):
        dist_str = "None" if item["distance"] is None else f"{item['distance']:.6f}"
        context_parts.append(
            f"[Reference {i+1}]\n"
            f"Source: {item['source']}\n"
            f"Verified label: {item['label']}\n"
            f"Similarity distance: {dist_str}\n"
            f"Text: {item['text']}\n"
        )

    context = "\n".join(context_parts)

    return f"""
You are a careful fake news detection assistant.

Your task is to classify the TARGET NEWS as fake or real.

You are given retrieved references with verified labels, but these references may be noisy or only superficially similar.
Do not trust them automatically.

Instructions:
1. For each reference, decide whether it is:
   - relevant_positive
   - relevant_negative
   - irrelevant
2. Only relevant_positive references can be used as strong evidence.
3. Ignore references that only overlap in person name, topic, wording, or style.
4. If evidence is weak or conflicting, make a cautious judgment based on the target text itself.
5. Prefer the strongest matching evidence, not the largest number of references.

Output exactly in this format:

analysis:
Reference 1: <relevant_positive / relevant_negative / irrelevant> - <short reason>
Reference 2: <relevant_positive / relevant_negative / irrelevant> - <short reason>
Reference 3: <relevant_positive / relevant_negative / irrelevant> - <short reason>
Reference 4: <relevant_positive / relevant_negative / irrelevant> - <short reason>
Reference 5: <relevant_positive / relevant_negative / irrelevant> - <short reason>
final reasoning: <brief paragraph>

answer: <fake or real>

Retrieved references:
{context}

Target news:
{query_text}
"""

# =========================================================
# 11. 输出解析
# =========================================================
def parse_prediction(output_text, retrieved_docs=None):
    text = output_text.strip().lower()

    match = re.search(r"answer\s*:\s*(fake|real)", text)
    if match:
        return 0 if match.group(1) == "fake" else 1

    if "fake" in text and "real" not in text:
        return 0
    if "real" in text and "fake" not in text:
        return 1

    fake_pos = text.find("fake") if "fake" in text else 10**9
    real_pos = text.find("real") if "real" in text else 10**9

    if fake_pos < real_pos:
        return 0
    if real_pos < fake_pos:
        return 1

    if retrieved_docs is not None:
        return fallback_by_retrieval(retrieved_docs)

    return 0

# =========================================================
# 12. 单条预测
# =========================================================
def predict_one_with_rag(news_text, top_k=5):
    retrieved_docs = retrieve_docs(news_text, top_k=top_k)
    prompt = build_rag_prompt(news_text[:2000], retrieved_docs)

    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(text, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=None,
            top_p=None
        )

    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    pred = parse_prediction(response, retrieved_docs)

    return response, pred, retrieved_docs

# =========================================================
# 13. 测试集评估
# =========================================================
y_true = []
y_pred = []
raw_outputs = []
retrieved_texts = []

for i, row in tqdm(test_df.iterrows(), total=len(test_df), desc="evaluating"):
    news_text = str(row["text"])
    true_label = int(row["label"])

    try:
        raw_output, pred_label, retrieved_docs = predict_one_with_rag(news_text, top_k=TOP_K)

        retrieved_joined = "\n\n".join([
            f"[{j+1}] label={d['label']}, source={d['source']}, distance={d['distance']}, text={d['text'][:300]}"
            for j, d in enumerate(retrieved_docs)
        ])
    except Exception as e:
        raw_output = f"ERROR: {e}"
        pred_label = 0
        retrieved_joined = ""

    y_true.append(true_label)
    y_pred.append(pred_label)
    raw_outputs.append(raw_output)
    retrieved_texts.append(retrieved_joined)

    print(f"\n[{i+1}/{len(test_df)}]")
    print("true =", true_label, "pred =", pred_label)
    print("raw_output =", raw_output[:500])

# =========================================================
# 14. 结果统计
# =========================================================
result_df = test_df.copy()
result_df["pred"] = y_pred
result_df["raw_output"] = raw_outputs
result_df["retrieved_docs"] = retrieved_texts

acc = accuracy_score(result_df["label"], result_df["pred"])
macro_f1 = f1_score(result_df["label"], result_df["pred"], average="macro")

print("\n总样本数:", len(result_df))
print("Accuracy:", acc)
print("Macro-F1:", macro_f1)
print("\nClassification Report:")
print(classification_report(result_df["label"], result_df["pred"], digits=4))

# =========================================================
# 15. 保存结果
# =========================================================
result_df.to_csv(SAVE_PATH, index=False)
print("\n结果已保存到:", SAVE_PATH)

torch: 2.11.0+cu130
cuda available: True
gpu: NVIDIA GeForce RTX 5090
train columns: ['sentence', 'label']
test columns: ['sentence', 'label']
train size: 645
test size: 628
train raw label dist:
raw_label
0    407
1    238
Name: count, dtype: int64
test raw label dist:
raw_label
0    314
1    314
Name: count, dtype: int64
train mapped label dist (0=fake,1=real):
label
1    407
0    238
Name: count, dtype: int64
test mapped label dist (0=fake,1=real):
label
1    314
0    314
Name: count, dtype: int64


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4787.39it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


embedding model device: cuda
embedding smoke test ok, shape: (1, 768)


OSError: Repo id must be in the form 'repo_name' or 'namespace/repo_name': '/root/autodl-tmp/models/hf_cache/Qwen2.5-7B-Instruct'. Use `repo_type` argument if needed.

In [ ]:
import os
import re
import shutil
import torch
import pandas as pd
import chromadb

from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# =========================================================
# 1. 路径配置
# =========================================================
ROOT_DIR = "/root/autodl-tmp"
DATASET_NAME = "allsides_3class_rag_heading_text"

CSV_PATH = "autodl-tmp/allsides_balanced_news_headlines-texts.csv"

OUTPUT_DIR = os.path.join(ROOT_DIR, DATASET_NAME)
DB_DIR = os.path.join(OUTPUT_DIR, "chroma_db")
COLLECTION_NAME = f"{DATASET_NAME}_bge"
SAVE_PATH = os.path.join(OUTPUT_DIR, "rag_results_test_first_1000.csv")

EMBED_MODEL_NAME = "BAAI/bge-base-en-v1.5"

# ===== 你的Qwen模型路径，自己按实际改 =====
QWEN_PATH = "/root/autodl-tmp/Qwen2.5-7B-Instruct"

TOP_K = 5
MAX_NEW_TOKENS = 512
BATCH_SIZE = 128
RANDOM_STATE = 42

TEST_RATIO = 0.1
TEST_FIRST_N = 1000   # 只测试 test_df 前 1000 个；如果不足 1000，则全测

os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================================================
# 2. 基本检查
# =========================================================
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("CUDA 不可用")
print("gpu:", torch.cuda.get_device_name(0))

# =========================================================
# 3. 读取数据（主文本列改为 heading + text 拼接）
# =========================================================
df = pd.read_csv(CSV_PATH)

print("columns:", df.columns.tolist())
print("raw label dist:")
print(df["bias_rating"].value_counts(dropna=False))

for col in ["heading", "text", "bias_rating"]:
    if col not in df.columns:
        raise ValueError(f"缺少列: {col}")

df["heading"] = df["heading"].fillna("").astype(str).str.strip()
df["text"] = df["text"].fillna("").astype(str).str.strip()

# 至少 heading 或 text 有一个非空
df = df[(df["heading"] != "") | (df["text"] != "")].reset_index(drop=True)

# 标签映射：0=left, 1=center, 2=right
label2id = {
    "left": 0,
    "center": 1,
    "right": 2
}
id2label = {v: k for k, v in label2id.items()}

df["bias_rating"] = df["bias_rating"].astype(str).str.strip().str.lower()
df = df[df["bias_rating"].isin(label2id.keys())].reset_index(drop=True)
df["label"] = df["bias_rating"].map(label2id).astype(int)

if "source" not in df.columns:
    df["source"] = "unknown_source"
else:
    df["source"] = df["source"].fillna("unknown_source").astype(str)

if "title" not in df.columns:
    df["title"] = ""
else:
    df["title"] = df["title"].fillna("").astype(str)

def combine_heading_text(heading: str, text: str) -> str:
    heading = (heading or "").strip()
    text = (text or "").strip()

    if heading and text:
        return f"Heading: {heading}\nText: {text}"
    if heading:
        return f"Heading: {heading}"
    if text:
        return f"Text: {text}"
    return ""

# 内部统一字段 text = heading + text
data = pd.DataFrame({
    "text": [combine_heading_text(h, t) for h, t in zip(df["heading"], df["text"])],
    "label": df["label"].astype(int),
    "source": df["source"].astype(str),
    "title": df["title"].astype(str),
    "heading": df["heading"].astype(str),
    "body_text": df["text"].astype(str),
    "bias_rating": df["bias_rating"].astype(str)
})

data = data[data["text"].str.strip() != ""].reset_index(drop=True)

print("clean size:", len(data))
print("mapped label dist:")
print(data["label"].value_counts())
print(data[["text", "label", "bias_rating"]].head())

# =========================================================
# 4. 9:1 分层切分
# =========================================================
train_df, test_df = train_test_split(
    data,
    test_size=TEST_RATIO,
    random_state=RANDOM_STATE,
    stratify=data["label"]
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

# 只取测试集前 1000 个样本
if TEST_FIRST_N is not None:
    test_df = test_df.iloc[:min(TEST_FIRST_N, len(test_df))].reset_index(drop=True)

print("train size:", len(train_df))
print("test size:", len(test_df))
print("train label dist:")
print(train_df["label"].value_counts())
print("test label dist:")
print(test_df["label"].value_counts())

# =========================================================
# 5. 加载 embedding 模型
# =========================================================
embed_model = SentenceTransformer(EMBED_MODEL_NAME, device="cuda")
print("embedding model device: cuda")

_test_emb = embed_model.encode(
    ["Represent this political news article for retrieving relevant ideological stance examples: Heading: test heading\nText: test article body"],
    normalize_embeddings=True,
    convert_to_numpy=True
)
print("embedding smoke test ok, shape:", _test_emb.shape)

# =========================================================
# 6. 加载 Qwen 4bit
# =========================================================
# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_compute_dtype=torch.float16,
#     bnb_4bit_use_double_quant=True,
#     bnb_4bit_quant_type="nf4"
# )

# tokenizer = AutoTokenizer.from_pretrained(
#     QWEN_PATH,
#     trust_remote_code=True
# )

# model = AutoModelForCausalLM.from_pretrained(
#     QWEN_PATH,
#     trust_remote_code=True,
#     device_map="auto",
#     quantization_config=bnb_config
# )

model.eval()
print("Qwen loaded")

# =========================================================
# 7. 建库（只用 train）
# 如果你已经建好了库，就直接 get_collection
# 如果没建好，把下面注释块打开重新建库
# =========================================================

# ===== 如果要重建库，取消下面这段注释 =====
# if os.path.exists(DB_DIR):
#     shutil.rmtree(DB_DIR)
#
# client = chromadb.PersistentClient(path=DB_DIR)
# collection = client.create_collection(
#     name=COLLECTION_NAME,
#     metadata={"hnsw:space": "cosine"}
# )
#
# texts = train_df["text"].tolist()
# labels = train_df["label"].tolist()
# sources = train_df["source"].tolist()
# titles = train_df["title"].tolist()
# headings = train_df["heading"].tolist()
# body_texts = train_df["body_text"].tolist()
# ids = [f"doc_{i}" for i in range(len(texts))]
#
# for start in tqdm(range(0, len(texts), BATCH_SIZE), desc="building chroma"):
#     end = min(start + BATCH_SIZE, len(texts))
#     batch_texts = texts[start:end]
#
#     batch_embeddings = embed_model.encode(
#         [f"Represent this political news article for retrieving relevant ideological stance examples: {x}" for x in batch_texts],
#         normalize_embeddings=True,
#         convert_to_numpy=True
#     ).tolist()
#
#     batch_metas = []
#     for lab, src, ttl, hd, bt in zip(
#         labels[start:end],
#         sources[start:end],
#         titles[start:end],
#         headings[start:end],
#         body_texts[start:end]
#     ):
#         batch_metas.append({
#             "label": int(lab),
#             "label_name": id2label[int(lab)],
#             "source": str(src),
#             "title": str(ttl),
#             "heading": str(hd),
#             "body_text": str(bt)
#         })
#
#     collection.add(
#         ids=ids[start:end],
#         documents=batch_texts,
#         embeddings=batch_embeddings,
#         metadatas=batch_metas
#     )
#
# print("collection count =", collection.count())

client = chromadb.PersistentClient(path=DB_DIR)
collection = client.get_collection(name=COLLECTION_NAME)
print("loaded collection:", COLLECTION_NAME)

# =========================================================
# 8. 检索函数
# =========================================================
def retrieve_docs(query_text, top_k=5):
    query_embedding = embed_model.encode(
        [f"Represent this political news article for retrieving relevant ideological stance examples: {query_text}"],
        normalize_embeddings=True,
        convert_to_numpy=True
    ).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k
    )

    docs = results["documents"][0]
    metas = results["metadatas"][0]
    dists = results["distances"][0] if "distances" in results else [None] * len(docs)

    retrieved = []
    for doc, meta, dist in zip(docs, metas, dists):
        retrieved.append({
            "text": doc,
            "label": meta.get("label_name", "unknown"),
            "source": meta.get("source", "unknown"),
            "title": meta.get("title", ""),
            "heading": meta.get("heading", ""),
            "body_text": meta.get("body_text", ""),
            "distance": dist
        })

    return retrieved

# =========================================================
# 9. 检索兜底
# =========================================================
def fallback_by_retrieval(retrieved_docs):
    if len(retrieved_docs) == 0:
        return 1  # 默认 center

    scores = {"left": 0.0, "center": 0.0, "right": 0.0}

    for d in retrieved_docs:
        dist = d["distance"] if d["distance"] is not None else 1.0
        weight = 1.0 / (dist + 1e-6)

        if d["label"] in scores:
            scores[d["label"]] += weight

    pred_label_name = max(scores.items(), key=lambda x: x[1])[0]
    return label2id[pred_label_name]

# =========================================================
# 10. Prompt（针对 heading + text）
# =========================================================
def build_rag_prompt(query_text, retrieved_docs):
    context_parts = []

    for i, item in enumerate(retrieved_docs):
        dist_str = "None" if item["distance"] is None else f"{item['distance']:.6f}"
        context_parts.append(
            f"[Reference {i+1}]\n"
            f"Source outlet: {item['source']}\n"
            f"Verified label: {item['label']}\n"
            f"Title: {item['title']}\n"
            f"Heading: {item['heading']}\n"
            f"Body text: {item['body_text'][:1500]}\n"
            f"Similarity distance: {dist_str}\n"
            f"Combined text: {item['text'][:2000]}\n"
        )

    context = "\n".join(context_parts)

    return f"""
You are a careful political media bias classification assistant.

Your task is to classify the TARGET ARTICLE into exactly one of these three labels:
- left
- center
- right

The target article contains a headline and body text together.

You are given retrieved references with verified labels, but these references may be noisy or only superficially similar.
Do not trust them automatically.

Instructions:
1. For each reference, decide whether it is:
   - relevant_positive
   - relevant_negative
   - irrelevant
2. Only relevant_positive references can be used as strong evidence.
3. Ignore references that overlap only in topic, named entities, or style, unless they are informative about ideological stance.
4. If evidence is weak or conflicting, make a cautious judgment based on the wording, framing, emphasis, and stance cues in the target article itself.
5. Prefer the strongest matching evidence, not the largest number of references.

Guidance:
- left: framing, emphasis, or language typically aligned with left-leaning political perspectives
- center: relatively neutral, balanced, or less ideologically marked presentation
- right: framing, emphasis, or language typically aligned with right-leaning political perspectives

Output exactly in this format:

analysis:
Reference 1: <relevant_positive / relevant_negative / irrelevant> - <short reason>
Reference 2: <relevant_positive / relevant_negative / irrelevant> - <short reason>
Reference 3: <relevant_positive / relevant_negative / irrelevant> - <short reason>
Reference 4: <relevant_positive / relevant_negative / irrelevant> - <short reason>
Reference 5: <relevant_positive / relevant_negative / irrelevant> - <short reason>
final reasoning: <brief paragraph>

answer: <left or center or right>

Retrieved references:
{context}

Target article:
{query_text}
"""

# =========================================================
# 11. 输出解析
# =========================================================
def parse_prediction(output_text, retrieved_docs=None):
    text = output_text.strip().lower()

    match = re.search(r"answer\s*:\s*(left|center|right)", text)
    if match:
        return label2id[match.group(1)]

    tail = "\n".join(text.splitlines()[-5:])
    for label_name in ["left", "center", "right"]:
        if re.search(rf"\b{label_name}\b", tail):
            return label2id[label_name]

    if retrieved_docs is not None:
        return fallback_by_retrieval(retrieved_docs)

    return 1  # 默认 center

# =========================================================
# 12. 单条预测
# =========================================================
def predict_one_with_rag(article_text, top_k=5):
    retrieved_docs = retrieve_docs(article_text, top_k=top_k)
    prompt = build_rag_prompt(article_text[:3000], retrieved_docs)

    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=8192
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=None,
            top_p=None
        )

    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    pred = parse_prediction(response, retrieved_docs)

    return response, pred, retrieved_docs

# =========================================================
# 13. 测试集评估
# =========================================================
y_true = []
y_pred = []
raw_outputs = []
retrieved_texts = []

for i, row in tqdm(test_df.iterrows(), total=len(test_df), desc="evaluating"):
    article_text = str(row["text"])
    true_label = int(row["label"])

    try:
        raw_output, pred_label, retrieved_docs = predict_one_with_rag(article_text, top_k=TOP_K)

        retrieved_joined = "\n\n".join([
            f"[{j+1}] label={d['label']}, source={d['source']}, distance={d['distance']}, "
            f"title={d['title'][:120]}, heading={d['heading'][:200]}, body_text={d['body_text'][:300]}"
            for j, d in enumerate(retrieved_docs)
        ])
    except Exception as e:
        raw_output = f"ERROR: {e}"
        pred_label = 1
        retrieved_joined = ""

    y_true.append(true_label)
    y_pred.append(pred_label)
    raw_outputs.append(raw_output)
    retrieved_texts.append(retrieved_joined)

    print(f"\n[{i+1}/{len(test_df)}]")
    print("true =", true_label, id2label[true_label], "| pred =", pred_label, id2label[pred_label])
    print("raw_output =")
    print(raw_output)
    print("-" * 100)

# =========================================================
# 14. 结果统计
# =========================================================
result_df = test_df.copy()
result_df["pred"] = y_pred
result_df["pred_name"] = result_df["pred"].map(id2label)
result_df["label_name"] = result_df["label"].map(id2label)
result_df["raw_output"] = raw_outputs
result_df["retrieved_docs"] = retrieved_texts

acc = accuracy_score(result_df["label"], result_df["pred"])
macro_f1 = f1_score(result_df["label"], result_df["pred"], average="macro")

print("\n总样本数:", len(result_df))
print("Accuracy:", acc)
print("Macro-F1:", macro_f1)
print("\nClassification Report:")
print(classification_report(
    result_df["label"],
    result_df["pred"],
    target_names=["left", "center", "right"],
    digits=4
))

# =========================================================
# 15. 保存结果
# =========================================================
result_df.to_csv(SAVE_PATH, index=False)
print("\n结果已保存到:", SAVE_PATH)

torch: 2.11.0+cu130
cuda available: True
gpu: NVIDIA GeForce RTX 5090
columns: ['Unnamed: 0', 'title', 'tags', 'heading', 'source', 'text', 'bias_rating']
raw label dist:
bias_rating
left      10275
right      7226
center     4253
Name: count, dtype: int64
clean size: 21754
mapped label dist:
label
0    10275
2     7226
1     4253
Name: count, dtype: int64
                                                text  label bias_rating
0  Heading: Chicago Gun Violence Spikes and Incre...      0        left
1  Heading: ‘Bullets just came from nowhere’: Fou...      1      center
2  Heading: Dozens of shootings across US mark bl...      2       right
3  Heading: Federal Government Will Run Out of Ca...      2       right
4  Heading: Yellen tells Congress that U.S. will ...      0        left
train size: 19578
test size: 1000
train label dist:
label
0    9247
2    6503
1    3828
Name: count, dtype: int64
test label dist:
label
0    466
2    353
1    181
Name: count, dtype: int64


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4084.37it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


embedding model device: cuda
embedding smoke test ok, shape: (1, 768)
Qwen loaded
loaded collection: allsides_3class_rag_heading_text_bge


evaluating:   0%|          | 1/1000 [00:04<1:11:48,  4.31s/it]


[1/1000]
true = 1 center | pred = 1 center
raw_output =
analysis:
Reference 1: irrelevant - The reference is about a different incident involving a carjacking and murder, focusing on the victim and the family's response. It does not provide direct evidence about the framing or stance of the target article.
Reference 2: irrelevant - This reference focuses on the same incident but provides more details about the carjacking and the suspects' actions. It does not offer specific information about the framing or stance of the target article.
Reference 3: irrelevant - This reference is about a completely different incident involving a car chase and a young child. It does not provide any relevant information about the target article.
Reference 4: irrelevant - This reference discusses a separate incident involving a shooting spree and does not relate to the carjacking and murder described in the target article.
Reference 5: irrelevant - This reference is about a different case involving a quad

...


[48/1000]
true = 0 left | pred = 0 left
raw_output =
analysis:
Reference 1: relevant_positive - The reference aligns with the target article's content, showing a shift in Biden's stance on the Hyde Amendment, which is a key issue for Democrats.
Reference 2: relevant_positive - This reference also describes Biden's reversal on the Hyde Amendment and provides context about the criticism he faced, matching the target article closely.
Reference 3: irrelevant - While this reference mentions Biden and the Hyde Amendment, it focuses on other issues like minimum wage and does not provide strong evidence for the target article's content.
Reference 4: irrelevant - This reference discusses Biden's discomfort with abortion and the leak of a Supreme Court draft opinion, which is not directly related to the target article's focus on the Hyde Amendment reversal.
Reference 5: irrelevant - This reference is about Biden's stance on impeachment and does not provide any relevant information about the Hyd

evaluating:   5%|▍         | 49/1000 [02:47<54:29,  3.44s/it]


[49/1000]
true = 1 center | pred = 0 left
raw_output =
analysis:
Reference 1: irrelevant - The reference is from Fox News and focuses on the relationship aspect rather than the framing or language.
Reference 2: relevant_positive - The reference from the New York Times aligns with the target article's framing and language, emphasizing Obama's condemnation of the anti-gay bill.
Reference 3: irrelevant - This reference discusses a different country (Kenya) and does not provide direct evidence for the target article.
Reference 4: irrelevant - This reference also discusses a different country (Kenya) and does not provide direct evidence for the target article.
Reference 5: irrelevant - This reference discusses a different issue (religious liberties in Indiana) and does not provide direct evidence for the target article.

final reasoning: The target article's framing and language closely align with the New York Times reference, which emphasizes Obama's condemnation of the anti-gay bill and 

evaluating:   5%|▌         | 50/1000 [02:50<54:36,  3.45s/it]


[50/1000]
true = 0 left | pred = 1 center
raw_output =
analysis:
Reference 1: irrelevant - The reference is about Mueller's testimony but does not provide specific ideological framing or emphasis.
Reference 2: irrelevant - While this reference mentions Mueller's testimony, it focuses more on his opening statement rather than providing clear ideological framing.
Reference 3: irrelevant - This reference discusses the scheduling of Mueller's testimony and the disagreement over the release of the report, without clear ideological framing.
Reference 4: irrelevant - This reference talks about the House Judiciary Committee's efforts to obtain the Mueller report, which is not directly related to the framing of Mueller's testimony.
Reference 5: irrelevant - This reference is about impeachment hearings, which are separate from Mueller's testimony and do not provide relevant context for the target article's framing.

final reasoning: The target article itself does not contain explicit ideologica

In [21]:
import os
import re
import shutil
import torch
import pandas as pd
import chromadb

from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# =========================================================
# 1. 路径配置
# =========================================================
ROOT_DIR = "/root/autodl-tmp"
DATASET_NAME = "merged_train_test_9_1_rag"

TRAIN_PATH = "/root/autodl-tmp/train.tsv"
TEST_PATH = "/root/autodl-tmp/test.tsv"

OUTPUT_DIR = os.path.join(ROOT_DIR, f"{DATASET_NAME}_output")
DB_DIR = os.path.join(OUTPUT_DIR, "chroma_db")
COLLECTION_NAME = f"{DATASET_NAME}_bge_large"
SAVE_PATH = os.path.join(OUTPUT_DIR, "rag_results.csv")

# 更强 embedding 模型
EMBED_MODEL_NAME = "BAAI/bge-large-en-v1.5"

# 你的 Qwen 路径
QWEN_PATH = "Qwen/Qwen2.5-7B-Instruct"
# 如果你本地已有模型目录，也可以改成：
# QWEN_PATH = "/root/autodl-tmp/你的模型目录"

TOP_K = 5
MAX_NEW_TOKENS = 500
BATCH_SIZE = 64
RANDOM_STATE = 42

os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================================================
# 2. 基本检查
# =========================================================
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("CUDA 不可用")
print("gpu:", torch.cuda.get_device_name(0))

# =========================================================
# 3. 读取 train / test，并合并
# =========================================================
train_raw = pd.read_csv(TRAIN_PATH, sep="\t")
test_raw = pd.read_csv(TEST_PATH, sep="\t")

print("train columns:", train_raw.columns.tolist())
print("test columns:", test_raw.columns.tolist())

required_cols = ["sentence", "label"]
for col in required_cols:
    if col not in train_raw.columns:
        raise ValueError(f"训练集缺少列: {col}")
    if col not in test_raw.columns:
        raise ValueError(f"测试集缺少列: {col}")

train_raw = train_raw[["sentence", "label"]].copy()
test_raw = test_raw[["sentence", "label"]].copy()

train_raw["orig_split"] = "train.tsv"
test_raw["orig_split"] = "test.tsv"

all_df = pd.concat([train_raw, test_raw], axis=0, ignore_index=True)

all_df = all_df.rename(columns={"sentence": "text", "label": "raw_label"})
all_df = all_df.dropna(subset=["text", "raw_label"]).reset_index(drop=True)
all_df["text"] = all_df["text"].astype(str).str.strip()
all_df = all_df[all_df["text"] != ""].reset_index(drop=True)
all_df["raw_label"] = all_df["raw_label"].astype(int)

print("merged total size:", len(all_df))
print("merged raw label dist:")
print(all_df["raw_label"].value_counts())

# =========================================================
# 4. 标签映射
# 内部统一：
#   0 = fake
#   1 = real
#
# 当前按你原代码逻辑：
#   原始 1 -> fake
#   原始 0 -> real
#
# 如果你确认相反，把 map_label 改掉
# =========================================================
def map_label(x: int) -> int:
    return 0 if x == 1 else 1

all_df["label"] = all_df["raw_label"].apply(map_label).astype(int)
all_df["label_name"] = all_df["label"].map({0: "fake", 1: "real"})

print("merged mapped label dist (0=fake,1=real):")
print(all_df["label"].value_counts())

# =========================================================
# 5. 合并后重新做 9:1 分层切分
# =========================================================
train_df, test_df = train_test_split(
    all_df,
    test_size=0.1,
    random_state=RANDOM_STATE,
    stratify=all_df["label"]
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

train_df["source"] = "merged_train_9"
test_df["source"] = "merged_test_1"

print("\nAfter merged 9:1 split")
print("train size:", len(train_df))
print("test size:", len(test_df))
print("train label dist:")
print(train_df["label"].value_counts())
print("test label dist:")
print(test_df["label"].value_counts())

# =========================================================
# 6. 加载 embedding 模型
# =========================================================
print(f"\nLoading embedding model: {EMBED_MODEL_NAME}")
embed_model = SentenceTransformer(EMBED_MODEL_NAME, device="cuda")
print("embedding model device: cuda")

EMBED_INSTRUCTION = "Represent this news article for retrieving relevant veracity examples: "

_test_emb = embed_model.encode(
    [EMBED_INSTRUCTION + "test sentence"],
    normalize_embeddings=True,
    convert_to_numpy=True,
    batch_size=1
)
print("embedding smoke test ok, shape:", _test_emb.shape)

# =========================================================
# 7. 加载 Qwen 4bit
# =========================================================
print(f"\nLoading Qwen model: {QWEN_PATH}")

# bnb_config = BitsAndBytesConfig(
#     load_in_4bit=True,
#     bnb_4bit_quant_type="nf4",
#     bnb_4bit_use_double_quant=True,
#     bnb_4bit_compute_dtype=torch.float16
# )

# tokenizer = AutoTokenizer.from_pretrained(
#     QWEN_PATH,
#     trust_remote_code=True
# )

# model = AutoModelForCausalLM.from_pretrained(
#     QWEN_PATH,
#     trust_remote_code=True,
#     device_map="auto",
#     quantization_config=bnb_config,
#     torch_dtype=torch.float16
# )

model.eval()
print("Qwen loaded")

# =========================================================
# 8. 建库（只用新的 train_df）
# =========================================================
# if os.path.exists(DB_DIR):
#     shutil.rmtree(DB_DIR)

# client = chromadb.PersistentClient(path=DB_DIR)
# collection = client.create_collection(
#     name=COLLECTION_NAME,
#     metadata={"hnsw:space": "cosine"}
# )

# texts = train_df["text"].tolist()
# labels = train_df["label"].tolist()
# sources = train_df["source"].tolist()
# orig_splits = train_df["orig_split"].tolist()
# ids = [f"doc_{i}" for i in range(len(texts))]

# for start in tqdm(range(0, len(texts), BATCH_SIZE), desc="building chroma"):
#     end = min(start + BATCH_SIZE, len(texts))
#     batch_texts = texts[start:end]

#     batch_embeddings = embed_model.encode(
#         [EMBED_INSTRUCTION + x for x in batch_texts],
#         normalize_embeddings=True,
#         convert_to_numpy=True,
#         batch_size=min(BATCH_SIZE, len(batch_texts)),
#         show_progress_bar=False
#     ).tolist()

#     batch_metas = []
#     for lab, src, osp in zip(labels[start:end], sources[start:end], orig_splits[start:end]):
#         batch_metas.append({
#             "label": int(lab),
#             "label_name": "fake" if int(lab) == 0 else "real",
#             "source": str(src),
#             "orig_split": str(osp)
#         })

#     collection.add(
#         ids=ids[start:end],
#         documents=batch_texts,
#         embeddings=batch_embeddings,
#         metadatas=batch_metas
#     )

# print("collection count =", collection.count())
client = chromadb.PersistentClient(path=DB_DIR)
collection = client.get_collection(name=COLLECTION_NAME)
# =========================================================
# 9. 检索
# =========================================================
def retrieve_docs(query_text, top_k=5):
    query_embedding = embed_model.encode(
        [EMBED_INSTRUCTION + query_text],
        normalize_embeddings=True,
        convert_to_numpy=True,
        batch_size=1,
        show_progress_bar=False
    ).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=top_k
    )

    docs = results["documents"][0]
    metas = results["metadatas"][0]
    dists = results["distances"][0] if "distances" in results else [None] * len(docs)

    retrieved = []
    for doc, meta, dist in zip(docs, metas, dists):
        retrieved.append({
            "text": doc,
            "label": meta.get("label_name", "unknown"),
            "source": meta.get("source", "unknown"),
            "orig_split": meta.get("orig_split", "unknown"),
            "distance": dist
        })

    return retrieved

# =========================================================
# 10. 检索兜底
# =========================================================
def fallback_by_retrieval(retrieved_docs):
    if len(retrieved_docs) == 0:
        return 0

    fake_score = 0.0
    real_score = 0.0

    for d in retrieved_docs:
        dist = d["distance"] if d["distance"] is not None else 1.0
        weight = 1.0 / (dist + 1e-6)

        if d["label"] == "fake":
            fake_score += weight
        elif d["label"] == "real":
            real_score += weight

    return 0 if fake_score >= real_score else 1

# =========================================================
# 11. Prompt（更短更稳）
def build_rag_prompt(query_text, retrieved_docs):
    context_parts = []

    for i, item in enumerate(retrieved_docs):
        dist_str = "None" if item["distance"] is None else f"{item['distance']:.6f}"
        context_parts.append(
            f"[Reference {i+1}]\n"
            f"Source outlet: {item['source']}\n"
            f"Verified label: {item['label']}\n"
            f"Original split: {item.get('orig_split', 'unknown')}\n"
            f"Similarity distance: {dist_str}\n"
            f"Reference text: {item['text'][:2000]}\n"
        )

    context = "\n".join(context_parts)

    return f"""
You are a careful fake news classification assistant.

Your task is to classify the TARGET NEWS into exactly one of these two labels:
- fake
- real

You are given retrieved references with verified labels, but these references may be noisy, only partially related, or retrieved because of keyword overlap.
Do not trust them automatically.

Instructions:
1. For each reference, decide whether it is:
   - relevant_supporting
   - relevant_opposing
   - irrelevant
2. A reference is relevant_supporting if it provides credible evidence or a highly similar claim pattern that supports the target label judgment.
3. A reference is relevant_opposing if it is clearly related but pushes against the likely label judgment for the target news.
4. A reference is irrelevant if it only overlaps in topic, names, events, or general style without giving useful evidence for veracity.
5. Do not rely on the majority label among retrieved references.
6. Use the TARGET NEWS itself as the primary evidence.
7. Focus on verifiability, specificity, internal consistency, credibility cues, sensationalism, misleading framing, and whether the claim looks fabricated, unverifiable, or trustworthy.
8. If the evidence is weak or mixed, make the most cautious judgment based mainly on the TARGET NEWS itself.
9. Prefer the strongest relevant evidence, not the largest number of references.

Guidance:
- fake: the text appears misleading, fabricated, unverifiable, sensationalized, internally inconsistent, or lacking credible grounding
- real: the text appears specific, coherent, plausible, and consistent with credible reporting style or verifiable claims

Output exactly in this format:

analysis:
Reference 1: <relevant_supporting / relevant_opposing / irrelevant> - <short reason>
Reference 2: <relevant_supporting / relevant_opposing / irrelevant> - <short reason>
Reference 3: <relevant_supporting / relevant_opposing / irrelevant> - <short reason>
Reference 4: <relevant_supporting / relevant_opposing / irrelevant> - <short reason>
Reference 5: <relevant_supporting / relevant_opposing / irrelevant> - <short reason>
final reasoning: <brief paragraph>

answer: <fake or real>

Retrieved references:
{context}

Target news:
{query_text}
"""

# =========================================================
# 12. 输出解析
# =========================================================
def parse_prediction(output_text, retrieved_docs=None):
    text = output_text.strip().lower()

    match = re.search(r"answer\s*:\s*(fake|real)", text)
    if match:
        return 0 if match.group(1) == "fake" else 1

    for label_name in ["fake", "real"]:
        if re.search(rf"\b{label_name}\b", text):
            return 0 if label_name == "fake" else 1

    if retrieved_docs is not None:
        return fallback_by_retrieval(retrieved_docs)

    return 0

# =========================================================
# 13. 单条预测
# =========================================================
SYSTEM_PROMPT = (
    "You are a precise fake news classifier. "
    "Return only the final label in the required format."
)

def predict_one_with_rag(news_text, top_k=5):
    retrieved_docs = retrieve_docs(news_text, top_k=top_k)
    user_prompt = build_rag_prompt(news_text[:2500], retrieved_docs)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=8192
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=None,
            top_p=None,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_ids = outputs[0][inputs["input_ids"].shape[1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
    pred = parse_prediction(response, retrieved_docs)

    return response, pred, retrieved_docs

# =========================================================
# 14. 测试集评估（用新的 1/10 test_df）
# =========================================================
y_true = []
y_pred = []
raw_outputs = []
retrieved_texts = []

for i, row in tqdm(test_df.iterrows(), total=len(test_df), desc="evaluating"):
    news_text = str(row["text"])
    true_label = int(row["label"])

    try:
        raw_output, pred_label, retrieved_docs = predict_one_with_rag(news_text, top_k=TOP_K)

        retrieved_joined = "\n\n".join([
            f"[{j+1}] label={d['label']}, source={d['source']}, orig_split={d['orig_split']}, "
            f"distance={d['distance']}, text={d['text'][:300]}"
            for j, d in enumerate(retrieved_docs)
        ])
    except Exception as e:
        raw_output = f"ERROR: {e}"
        pred_label = 0
        retrieved_joined = ""

    y_true.append(true_label)
    y_pred.append(pred_label)
    raw_outputs.append(raw_output)
    retrieved_texts.append(retrieved_joined)

    print(f"\n[{i+1}/{len(test_df)}]")
    print("true =", true_label, "pred =", pred_label)
    print("raw_output =", raw_output[:300])

# =========================================================
# 15. 结果统计
# =========================================================
result_df = test_df.copy()
result_df["pred"] = y_pred
result_df["pred_name"] = result_df["pred"].map({0: "fake", 1: "real"})
result_df["label_name"] = result_df["label"].map({0: "fake", 1: "real"})
result_df["raw_output"] = raw_outputs
result_df["retrieved_docs"] = retrieved_texts

acc = accuracy_score(result_df["label"], result_df["pred"])
macro_f1 = f1_score(result_df["label"], result_df["pred"], average="macro")

print("\n总样本数:", len(result_df))
print("Accuracy:", acc)
print("Macro-F1:", macro_f1)
print("\nClassification Report:")
print(classification_report(
    result_df["label"],
    result_df["pred"],
    target_names=["fake", "real"],
    digits=4
))

# =========================================================
# 16. 保存结果
# =========================================================
result_df.to_csv(SAVE_PATH, index=False)
print("\n结果已保存到:", SAVE_PATH)

torch: 2.11.0+cu130
cuda available: True
gpu: NVIDIA GeForce RTX 5090
train columns: ['sentence', 'label']
test columns: ['sentence', 'label']
merged total size: 1273
merged raw label dist:
raw_label
0    721
1    552
Name: count, dtype: int64
merged mapped label dist (0=fake,1=real):
label
1    721
0    552
Name: count, dtype: int64

After merged 9:1 split
train size: 1145
test size: 128
train label dist:
label
1    649
0    496
Name: count, dtype: int64
test label dist:
label
1    72
0    56
Name: count, dtype: int64

Loading embedding model: BAAI/bge-large-en-v1.5


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 7255.71it/s]
BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


embedding model device: cuda
embedding smoke test ok, shape: (1, 1024)

Loading Qwen model: Qwen/Qwen2.5-7B-Instruct
Qwen loaded


evaluating:   1%|          | 1/128 [00:03<08:02,  3.80s/it]


[1/128]
true = 0 pred = 1
raw_output = analysis:
Reference 1: irrelevant - The reference discusses America's repudiation of the president following the Charlottesville incident, which is not directly related to the grammar debate in the target news.
Reference 2: irrelevant - The reference criticizes Trump's presidency and compares it unf


...


[126/128]
true = 1 pred = 1
raw_output = analysis:
Reference 1: irrelevant - The reference discusses a different incident involving a boy on a bicycle and does not provide evidence for the target news.
Reference 2: irrelevant - The reference is about a fictional story and does not provide any evidence for the target news.
Reference 3: irre


evaluating:  99%|█████████▉| 127/128 [07:19<00:03,  3.46s/it]


[127/128]
true = 0 pred = 1
raw_output = analysis:
Reference 1: irrelevant - The reference discusses unrelated incidents involving police and racism, providing no direct evidence for the target news.
Reference 2: irrelevant - This reference is about police procedures and does not relate to the surveillance video issue.
Reference 3: irrelev


evaluating: 100%|██████████| 128/128 [07:22<00:00,  3.46s/it]


[128/128]
true = 1 pred = 1
raw_output = analysis:
Reference 1: relevant_supporting - Provides context about Trump's response to investigations, supporting the real nature of political investigations.
Reference 2: irrelevant - Discusses student reactions to border security, unrelated to the news about misused taxpayer funds.
Reference 3: i

总样本数: 128
Accuracy: 0.625
Macro-F1: 0.5797537619699042

Classification Report:
              precision    recall  f1-score   support

        fake     0.6333    0.3393    0.4419        56
        real     0.6224    0.8472    0.7176        72

    accuracy                         0.6250       128
   macro avg     0.6279    0.5933    0.5798       128
weighted avg     0.6272    0.6250    0.5970       128


结果已保存到: /root/autodl-tmp/merged_train_test_9_1_rag_output/rag_results.csv
